# CAS-Match: Reliability-Aware Pseudo-Label Admission

**Standalone experiment notebook.** ConvNeXt-Tiny, CRIC/RIVA, 5%/10% labeled budgets. This notebook does not modify or write to Phase 11.

## Experiment matrix

Five methods per setting (20 runs total):

1. `fixmatch`: confidence-only, fixed threshold 0.95.
2. `lower_only`: class-specific lower threshold, confidence-only low route.
3. `agreement_only`: lower route requires weak/strong agreement.
4. `stability_only`: lower route requires temporal class stability.
5. `cas_match`: lower route requires both agreement and stability.

All methods share the same fixed V2 splits, stratified labeled subsets, initialization seed protocol, optimizer, schedule, and new augmentation pipeline. Validation labels are used only for per-run threshold calibration and checkpoint selection. Test labels are loaded only for final evaluation after training is complete.

**Calibration protocol:** after three warm-up epochs, build validation prediction history from the three prior epochs. For each predicted class and each candidate lower threshold (0.60, 0.70, 0.80), evaluate the *method-specific* low-route admission condition on validation. Select the lowest threshold with at least 90% pseudo-label precision and at least `MIN_CALIBRATION_SUPPORT` accepted validation samples for that predicted class. If none qualifies, use 0.80 and record failure to meet the target. Thresholds are frozen thereafter. Validation samples never enter gradient updates.

The 90% target and minimum support are experimental design choices, not universal constants. The notebook records calibration counts and precision for auditability.

In [1]:
# 1. Imports and configuration
from pathlib import Path
import os, json, time, random, hashlib, math, copy, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, balanced_accuracy_score)

SEED = 42
DATASETS = ("cric", "riva")
FRACTIONS = (0.05, 0.10)
METHODS = ("fixmatch", "lower_only", "agreement_only", "stability_only", "cas_match")
IMAGE_SIZE = 128
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
TAU_HIGH = 0.95
TAU_CANDIDATES = (0.60, 0.70, 0.80)
PRECISION_TARGET = 0.90
MIN_CALIBRATION_SUPPORT = 30  # configurable; required per predicted class and candidate
MEMORY_K = 5
MEMORY_MIN = 3
STABILITY_MIN = 0.80
UNSUP_WEIGHT = 1.0
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

# Set ROOT explicitly if automatic discovery fails.
ROOT = None
_candidates = [Path.cwd(), Path.cwd().parent, Path.home() / "Documents" / "cervical-semi-sl"]
if ROOT is None:
    ROOT = next((p.resolve() for p in _candidates if (p / "data" / "splits_v2" / "cric_splits_v2.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Project root not found. Set ROOT to your cervical-semi-sl folder.")
ROOT = Path(ROOT)
SPLIT_DIR = ROOT / "data" / "splits_v2"
OUT_DIR = ROOT / "data" / "results" / "cas_match_v1"  # separate from Phase 11
OUT_DIR.mkdir(parents=True, exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

CONFIG = {"seed":SEED, "datasets":DATASETS, "fractions":FRACTIONS, "methods":METHODS,
          "backbone":"convnext_tiny", "image_size":IMAGE_SIZE, "batch_size":BATCH_SIZE,
          "max_epochs":MAX_EPOCHS, "patience":PATIENCE, "lr":LR,
          "weight_decay":WEIGHT_DECAY, "tau_high":TAU_HIGH,
          "tau_candidates":TAU_CANDIDATES, "precision_target":PRECISION_TARGET,
          "min_calibration_support":MIN_CALIBRATION_SUPPORT, "memory_k":MEMORY_K,
          "memory_min":MEMORY_MIN, "stability_min":STABILITY_MIN,
          "unsup_weight":UNSUP_WEIGHT}
# Include augmentation choices in the resume identity. The split-file hashes are
# added after manifests are loaded in Cell 3.
AUGMENTATION_CONFIG = {
    "weak": {"horizontal_flip_p": 0.5, "rotation_degrees": 8},
    "strong": {"random_resized_crop_scale": (0.80, 1.0), "ratio": (0.90, 1.10),
               "color_jitter": (0.15, 0.15, 0.10, 0.02), "cutout_fraction": 0.25},
    "normalization_mean": MEAN, "normalization_std": STD,
}
CONFIG["augmentations"] = AUGMENTATION_CONFIG
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("ROOT:", ROOT, "\nOutput:", OUT_DIR, "\nDevice:", DEVICE, "AMP:", AMP)
print("Configuration hash:", CONFIG_HASH)

ROOT: C:\Users\nanda\Documents\cervical-semi-sl 
Output: C:\Users\nanda\Documents\cervical-semi-sl\data\results\cas_match_v1 
Device: cuda AMP: True
Configuration hash: 01dcebbd098d188d


## 2. Manifest validation and fixed labeled subsets

The manifest schema is expected to contain `crop_path`, `binary_label`, and `split`. Existing V2 splits are reused without modification. Labeled subsets are deterministic and shared across all five methods within a dataset/budget setting.

In [2]:
def resolve_crop(p):
    p = Path(str(p))
    candidates = [p] if p.is_absolute() else [ROOT / p, ROOT / "data" / p]
    for c in candidates:
        if c.exists(): return c.resolve()
    return None

manifests = {}
for ds in DATASETS:
    csv_path = SPLIT_DIR / f"{ds}_splits_v2.csv"
    if not csv_path.exists(): raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    required = {"crop_path", "binary_label", "split"}
    if required - set(df.columns): raise ValueError(f"{ds}: missing {required-set(df.columns)}")
    df = df.copy()
    df["split"] = df["split"].astype(str).str.strip().str.lower().replace({"validation":"val", "valid":"val"})
    if set(df["split"].unique()) != {"train", "val", "test"}:
        raise ValueError(f"{ds}: expected train/val/test, got {sorted(df.split.unique())}")
    df["binary_label"] = pd.to_numeric(df["binary_label"], errors="raise").astype(int)
    if not set(df.binary_label.unique()).issubset({0,1}): raise ValueError("Labels must be 0/1")
    df["resolved_path"] = df.crop_path.map(resolve_crop)
    if df.resolved_path.isna().any(): raise FileNotFoundError(f"Unresolved paths in {ds}: {df[df.resolved_path.isna()].crop_path.head().tolist()}")
    if df.resolved_path.duplicated().any(): raise ValueError(f"Duplicate crop paths in {ds}")
    manifests[ds] = df.reset_index(drop=True)
    print(f"\n{ds.upper()} rows={len(df):,}")
    display(pd.crosstab(df.split, df.binary_label, margins=True))

# Resume must not silently reuse checkpoints if either split manifest changes.
_manifest_hashes = {ds: hashlib.sha256((SPLIT_DIR / f"{ds}_splits_v2.csv").read_bytes()).hexdigest()
                    for ds in DATASETS}
CONFIG["manifest_sha256"] = _manifest_hashes
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("Final resume configuration hash (includes augmentations + split manifests):", CONFIG_HASH)

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def labeled_frames(ds, frac):
    train = manifests[ds].query("split == 'train'").reset_index(drop=True)
    subset_dir = OUT_DIR / "label_subsets"; subset_dir.mkdir(parents=True, exist_ok=True)
    subset_path = subset_dir / f"{ds}_labeled_{int(frac*100)}pct_seed{SEED}.csv"
    if subset_path.exists():
        saved = pd.read_csv(subset_path); wanted = set(saved.resolved_path.astype(str))
        lab = train[train.resolved_path.astype(str).isin(wanted)].copy()
        if len(lab) != len(wanted): raise ValueError(f"Saved subset mismatch: {subset_path}")
    else:
        rng = np.random.default_rng(SEED + (0 if ds == "cric" else 10000) + int(frac*100))
        picked=[]
        for cls in (0,1):
            idx=np.flatnonzero(train.binary_label.to_numpy()==cls)
            n=max(1,int(round(frac*len(idx))))
            picked.extend(rng.choice(idx,size=min(n,len(idx)),replace=False).tolist())
        lab=train.iloc[sorted(picked)].copy()
        lab[["resolved_path","binary_label"]].to_csv(subset_path,index=False)
    unlab=train[~train.resolved_path.isin(lab.resolved_path)].copy()
    return lab.reset_index(drop=True), unlab.reset_index(drop=True)

for ds in DATASETS:
    for frac in FRACTIONS:
        lab, unlab = labeled_frames(ds, frac)
        print(f"{ds.upper()} {frac:.0%}: labeled={len(lab):,}, unlabeled={len(unlab):,}")


CRIC rows=11,534


binary_label,0,1,All
split,,,
test,1603,704,2307
train,4482,3592,8074
val,694,459,1153
All,6779,4755,11534



RIVA rows=15,949


binary_label,0,1,All
split,,,
test,2070,1126,3196
train,7332,3839,11171
val,1115,467,1582
All,10517,5432,15949


Final resume configuration hash (includes augmentations + split manifests): e213e0fc343fe38f
CRIC 5%: labeled=404, unlabeled=7,670
CRIC 10%: labeled=807, unlabeled=7,267
RIVA 5%: labeled=559, unlabeled=10,612
RIVA 10%: labeled=1,117, unlabeled=10,054


## 3. Augmentations and datasets

The new controlled comparison uses the diagram's augmentation design. Weak augmentation is horizontal flip plus small rotation. Strong augmentation is random crop, color jitter, and cutout. The labeled branch uses the weak geometric pipeline for consistency with the diagram. All inputs are resized to 128×128 and ImageNet-normalized.

In [3]:
class Cutout:
    def __init__(self, fraction=0.25): self.fraction = fraction
    def __call__(self, tensor):
        # Applied after ToTensor and before normalization only in the strong transform.
        _, h, w = tensor.shape
        ch, cw = max(1, int(h*self.fraction)), max(1, int(w*self.fraction))
        top = random.randint(0, max(0, h-ch)); left = random.randint(0, max(0, w-cw))
        out = tensor.clone(); out[:, top:top+ch, left:left+cw] = 0.0
        return out

weak_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
strong_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(), Cutout(0.25), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class CellDataset(Dataset):
    def __init__(self, frame, mode): self.frame=frame.reset_index(drop=True); self.mode=mode
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r=self.frame.iloc[i]
        with Image.open(r.resolved_path) as im: im=im.convert("RGB")
        if self.mode == "unlabeled":
            return weak_tf(im), strong_tf(im), i
        if self.mode == "train": return weak_tf(im), int(r.binary_label)
        if self.mode == "valcal": return eval_tf(im), int(r.binary_label), i
        return eval_tf(im), int(r.binary_label)

def make_loader(frame, mode, shuffle=False, batch_size=BATCH_SIZE, seed=SEED):
    gen=torch.Generator().manual_seed(seed)
    return DataLoader(CellDataset(frame,mode), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=AMP, drop_last=False, generator=gen)

## 4. Model, metrics, and temporal memory

In [4]:
def build_model():
    m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2]=nn.Linear(m.classifier[2].in_features,2)
    return m.to(DEVICE)

CLASS_NAMES={0:"Normal",1:"Abnormal"}
def calc_metrics(y, prob):
    y=np.asarray(y,dtype=int); prob=np.asarray(prob,dtype=float); pred=(prob>=0.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    try: auc=float(roc_auc_score(y,prob))
    except ValueError: auc=float("nan")
    return {"accuracy":float(accuracy_score(y,pred)), "precision_abnormal":float(precision_score(y,pred,zero_division=0)),
            "abnormal_recall":float(recall_score(y,pred,zero_division=0)),
            "specificity_normal":float(tn/(tn+fp)) if tn+fp else float("nan"),
            "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "balanced_accuracy":float(balanced_accuracy_score(y,pred)), "auroc":auc,
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
            "normal_f1":float(f1_score(y,pred,labels=[0],average="macro",zero_division=0)),
            "abnormal_f1":float(f1_score(y,pred,labels=[1],average="macro",zero_division=0))}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys=[]; probs=[]
    for x,y in loader:
        x=x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP): logits=model(x)
        probs.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()); ys.extend(y.numpy())
    return calc_metrics(ys,probs)

class PredictionMemory:
    """Per-image circular buffer of hard classes. Current prediction is written after mask calculation."""
    def __init__(self,n_images,k=MEMORY_K):
        self.values=np.full((n_images,k),-1,dtype=np.int8); self.count=np.zeros(n_images,dtype=np.int16); self.pos=np.zeros(n_images,dtype=np.int16); self.k=k
    def stability(self, ids, current_pred):
        ids=np.asarray(ids,dtype=int); current_pred=np.asarray(current_pred,dtype=int)
        scores=np.zeros(len(ids),dtype=np.float32); counts=self.count[ids].copy()
        for j,(idx,pred) in enumerate(zip(ids,current_pred)):
            n=int(self.count[idx])
            if n: scores[j]=float(np.mean(self.values[idx,:n] == pred))
        return scores, counts
    def write(self, ids, pred):
        for idx,p in zip(np.asarray(ids,dtype=int),np.asarray(pred,dtype=int)):
            slot=int(self.pos[idx]); self.values[idx,slot]=int(p)
            self.pos[idx]=(slot+1)%self.k; self.count[idx]=min(self.k,int(self.count[idx])+1)

def admission_mask(method, conf, weak_pred, strong_pred, stability, history_count, tau_by_class):
    """Returns mask and route labels. All decisions use detached weak/strong predictions."""
    conf=np.asarray(conf); weak_pred=np.asarray(weak_pred); strong_pred=np.asarray(strong_pred)
    stability=np.asarray(stability); history_count=np.asarray(history_count)
    high=conf >= TAU_HIGH
    if method == "fixmatch": return high, np.where(high,"high","reject")
    low_band=np.array([conf[i] >= tau_by_class[int(c)] and conf[i] < TAU_HIGH for i,c in enumerate(weak_pred)])
    agree=(weak_pred==strong_pred)
    stable=(stability>=STABILITY_MIN)&(history_count>=MEMORY_MIN)
    if method == "lower_only": extra=np.ones_like(high,dtype=bool)
    elif method == "agreement_only": extra=agree
    elif method == "stability_only": extra=stable
    elif method == "cas_match": extra=agree & stable
    else: raise ValueError(method)
    low=low_band & extra
    mask=high | low
    route=np.where(high,"high",np.where(low,"low", "reject"))
    return mask,route

## 5. Validation threshold calibration

Validation is treated as unlabeled for prediction. Its labels are used offline only to score pseudo-label precision. The validation history is built from deterministic evaluation views over the three warm-up epochs. Calibration occurs at the start of epoch 4 using only the previous history, and thresholds remain frozen for the rest of training.

For each predicted class, candidate precision is computed among samples in the low-confidence band that satisfy the method-specific admission condition. A candidate must meet both the 90% precision target and minimum support. The lowest qualifying candidate is selected; otherwise 0.80 is selected and flagged. This procedure is run separately for every method and setting.

In [5]:
@torch.no_grad()
def predict_validation(model, val_df):
    model.eval(); dl=make_loader(val_df,"valcal",shuffle=False,batch_size=BATCH_SIZE)
    probs=[]; ys=[]; ids=[]
    for x,y,i in dl:
        x=x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP): logits=model(x)
        probs.append(torch.softmax(logits.float(),dim=1).cpu().numpy()); ys.extend(y.numpy()); ids.extend(i.numpy())
    return np.concatenate(probs), np.asarray(ys), np.asarray(ids)

def calibrate_thresholds(method, probs, true_labels, val_memory):
    pred=probs.argmax(1); conf=probs.max(1)
    # Validation memory contains only predictions from preceding warm-up epochs.
    stability, hist_count=val_memory.stability(np.arange(len(pred)),pred)
    strong_pred=pred.copy()  # no stochastic strong view during deterministic calibration; replaced below for agreement methods
    # The caller supplies validation strong-view predictions in probs_strong when available.
    rows=[]; selected={}; flags={}
    for cls in (0,1):
        chosen=None
        for tau in TAU_CANDIDATES:
            band=(pred==cls)&(conf>=tau)&(conf<TAU_HIGH)
            agree=(pred==strong_pred)
            stable=(stability>=STABILITY_MIN)&(hist_count>=MEMORY_MIN)
            if method in ("agreement_only","cas_match"): band &= agree
            if method in ("stability_only","cas_match"): band &= stable
            n=int(band.sum()); precision=float((pred[band]==true_labels[band]).mean()) if n else float("nan")
            meets=bool(n>=MIN_CALIBRATION_SUPPORT and precision>=PRECISION_TARGET)
            rows.append({"method":method,"predicted_class":cls,"candidate_tau":tau,"accepted_n":n,
                         "precision":precision,"target_met":meets})
            if meets and chosen is None: chosen=tau
        selected[cls]=float(chosen if chosen is not None else max(TAU_CANDIDATES))
        flags[cls]=bool(chosen is not None)
    return selected, flags, pd.DataFrame(rows)

@torch.no_grad()
def predict_val_two_views(model,val_df):
    model.eval(); weak_probs=[]; strong_probs=[]; ys=[]
    # Use actual weak/strong transforms for agreement; labels are collected strictly for post-hoc scoring.
    ds=CellDataset(val_df,"unlabeled")
    dl=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    for w,s,_ in dl:
        w=w.to(DEVICE); s=s.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP):
            pw=torch.softmax(model(w).float(),dim=1); ps=torch.softmax(model(s).float(),dim=1)
        weak_probs.append(pw.cpu().numpy()); strong_probs.append(ps.cpu().numpy())
    ys=val_df.binary_label.to_numpy(dtype=int)
    return np.concatenate(weak_probs),np.concatenate(strong_probs),ys

def calibrate_thresholds(method, weak_probs, strong_probs, true_labels, val_memory):
    pred=weak_probs.argmax(1); conf=weak_probs.max(1); strong_pred=strong_probs.argmax(1)
    stability,hist_count=val_memory.stability(np.arange(len(pred)),pred)
    rows=[]; selected={}; flags={}
    for cls in (0,1):
        chosen=None
        for tau in TAU_CANDIDATES:
            band=(pred==cls)&(conf>=tau)&(conf<TAU_HIGH)
            if method in ("agreement_only","cas_match"): band &= (pred==strong_pred)
            if method in ("stability_only","cas_match"):
                band &= (stability>=STABILITY_MIN)&(hist_count>=MEMORY_MIN)
            n=int(band.sum()); precision=float((pred[band]==true_labels[band]).mean()) if n else float("nan")
            meets=bool(n>=MIN_CALIBRATION_SUPPORT and precision>=PRECISION_TARGET)
            rows.append({"method":method,"predicted_class":cls,"candidate_tau":tau,
                         "accepted_validation_n":n,"pseudo_label_precision":precision,"target_met":meets})
            if meets and chosen is None: chosen=tau
        selected[cls]=float(chosen if chosen is not None else max(TAU_CANDIDATES))
        flags[cls]=bool(chosen is not None)
    return selected,flags,pd.DataFrame(rows)

## 6. Training utilities and reproducibility

The unlabeled sampler visits every unlabeled image once per epoch (last partial batch retained), allowing one memory update per image per epoch. Validation history is separate from unlabeled-training history. The high-confidence route is active from epoch 1; low-confidence admission activates only when three prior predictions exist, i.e. from epoch 4.

In [6]:
def stable_seed(text): return int(hashlib.sha256(text.encode()).hexdigest()[:8],16)
def run_name(ds,frac,method): return f"{ds}_{int(frac*100)}pct_{method}_seed{SEED}"

def make_unlabeled_epoch_loader(frame, epoch, run_id):
    # A deterministic shuffled order per epoch; all samples are visited exactly once.
    gen=torch.Generator().manual_seed(SEED+stable_seed(run_id)+epoch)
    ds=CellDataset(frame,"unlabeled")
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=AMP,drop_last=False,generator=gen)

def save_checkpoint(path, model, optimizer, scaler, epoch, best_score, bad_epochs, history,
                    train_memory, val_memory, tau_by_class, calibration_done):
    state={"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),
           "epoch":epoch,"best_score":best_score,"bad_epochs":bad_epochs,"history":history,
           "train_memory":train_memory.__dict__,"val_memory":val_memory.__dict__,
           "tau_by_class":tau_by_class,"calibration_done":calibration_done,
           "python_rng":random.getstate(),"numpy_rng":np.random.get_state(),"torch_rng":torch.get_rng_state(),
           "cuda_rng":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
           "config_hash":CONFIG_HASH}
    tmp=path.with_suffix(".tmp"); torch.save(state,tmp); os.replace(tmp,path)

def restore_memory(obj,state):
    obj.values=state["values"]; obj.count=state["count"]; obj.pos=state["pos"]; obj.k=state["k"]

def load_checkpoint(path,model,optimizer,scaler,train_memory,val_memory):
    st=torch.load(path,map_location=DEVICE,weights_only=False)
    if st.get("config_hash")!=CONFIG_HASH: raise ValueError("Checkpoint config hash mismatch; do not resume with changed settings")
    model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"]); scaler.load_state_dict(st["scaler"])
    restore_memory(train_memory,st["train_memory"]); restore_memory(val_memory,st["val_memory"])
    random.setstate(st["python_rng"]); np.random.set_state(st["numpy_rng"]); torch.set_rng_state(st["torch_rng"].cpu())
    if torch.cuda.is_available() and st.get("cuda_rng") is not None: torch.cuda.set_rng_state_all(st["cuda_rng"])
    return st

## 7. One experiment

Training uses a labeled batch and an unlabeled batch. The same model processes weak and strong views. The pseudo-label and admission mask are detached. Unsupervised loss is normalized by the **full unlabeled batch size**, matching the FixMatch convention. Pseudo-label diagnostics are accumulated from the actual training views; unlabeled ground-truth labels are not accessed by the training loop.

In [7]:
def train_experiment(ds,frac,method, max_epochs=None):
    max_epochs=MAX_EPOCHS if max_epochs is None else int(max_epochs)
    run_id=run_name(ds,frac,method); rd=OUT_DIR/run_id; rd.mkdir(parents=True,exist_ok=True)
    latest=rd/"checkpoint_latest.pt"; best=rd/"checkpoint_best.pt"
    done=rd/"DONE.json"
    if done.exists():
        result_path=rd/"test_metrics.json"
        if result_path.exists():
            print("Already completed:",run_id)
            return json.loads(result_path.read_text())
        # A completion marker without its result is inconsistent; do not silently skip it.
        raise RuntimeError(f"{run_id} has DONE.json but no test_metrics.json; inspect {rd}")
    lab,unlab=labeled_frames(ds,frac)
    val_df=manifests[ds].query("split == 'val'").reset_index(drop=True)
    test_df=manifests[ds].query("split == 'test'").reset_index(drop=True)
    lab_ds=CellDataset(lab,"train")
    val_eval=make_loader(val_df,"eval",shuffle=False)
    seed_all(SEED+stable_seed(f"{ds}_{frac}")%100000)
    model=build_model()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler("cuda",enabled=AMP)
    train_memory=PredictionMemory(len(unlab),MEMORY_K)
    val_memory=PredictionMemory(len(val_df),MEMORY_K)
    tau_by_class={0:TAU_HIGH,1:TAU_HIGH}; calibration_done=(method=="fixmatch")
    calibration_rows=[]; history=[]; best_score=-1.; bad_epochs=0; start_epoch=0
    if latest.exists():
        st=load_checkpoint(latest,model,optimizer,scaler,train_memory,val_memory)
        start_epoch=st["epoch"]+1; best_score=st["best_score"]; bad_epochs=st["bad_epochs"]
        history=st["history"]; tau_by_class=st["tau_by_class"]; calibration_done=st["calibration_done"]
        cal_path=rd/"threshold_calibration.csv"
        if cal_path.exists(): calibration_rows=pd.read_csv(cal_path).to_dict("records")
        print("Resuming",run_id,"at epoch",start_epoch+1)

    for epoch in range(start_epoch,max_epochs):
        # At start of epoch 4, validation history contains predictions from epochs 1-3.
        if method!="fixmatch" and epoch==MEMORY_MIN and not calibration_done:
            wp,sp,vy=predict_val_two_views(model,val_df)
            tau_by_class,cal_flags,cal_df=calibrate_thresholds(method,wp,sp,vy,val_memory)
            cal_df["selected_tau_class"] = cal_df.predicted_class.map(tau_by_class)
            cal_df["class_target_achieved"] = cal_df.predicted_class.map(cal_flags)
            cal_df.to_csv(rd/"threshold_calibration.csv",index=False)
            calibration_done=True
            print("Frozen class thresholds:",tau_by_class,"target achieved:",cal_flags)

        model.train(); epoch_losses=[]; diag=[]
        # labeled batches shuffled deterministically each epoch
        gen=torch.Generator().manual_seed(SEED+stable_seed(run_id)+epoch+777)
        perm=torch.randperm(len(lab_ds),generator=gen).tolist()
        lab_batches=[perm[i:i+BATCH_SIZE] for i in range(0,len(perm),BATCH_SIZE)]
        unl_loader=make_unlabeled_epoch_loader(unlab,epoch,run_id)
        # Drive the epoch by the unlabeled pool so every unlabeled image receives
        # exactly one memory observation per epoch. Labeled batches cycle as needed.
        for bi,(uw,us,uids) in enumerate(unl_loader):
            idxs=lab_batches[bi % len(lab_batches)]
            x=torch.stack([lab_ds[j][0] for j in idxs]).to(DEVICE)
            y=torch.tensor([lab_ds.frame.iloc[j].binary_label for j in idxs],dtype=torch.long,device=DEVICE)
            uw=uw.to(DEVICE); us=us.to(DEVICE); uids_np=uids.numpy().astype(int)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                logits_l=model(x); loss_s=F.cross_entropy(logits_l,y)
                with torch.no_grad():
                    pw=torch.softmax(model(uw).float(),dim=1)
                    conf,pseudo=pw.max(dim=1)
                    ps_pred=model(us).float().argmax(dim=1)
                    stab,hcount=train_memory.stability(uids_np,pseudo.cpu().numpy())
                    mask_np,route=admission_mask(method,conf.cpu().numpy(),pseudo.cpu().numpy(),
                                                  ps_pred.cpu().numpy(),stab,hcount,tau_by_class)
                    mask=torch.as_tensor(mask_np,dtype=torch.bool,device=DEVICE)
                logits_u=model(us)
                per_sample=F.cross_entropy(logits_u,pseudo.detach(),reduction="none")
                loss_u=(per_sample*mask.float()).mean()  # full unlabeled batch denominator
                loss=loss_s+UNSUP_WEIGHT*loss_u
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            # Memory write happens after mask calculation and optimizer update.
            train_memory.write(uids_np,pseudo.cpu().numpy())
            epoch_losses.append(float(loss.detach().cpu()))
            pred_np=pseudo.cpu().numpy(); mask_np=np.asarray(mask_np); route=np.asarray(route)
            # Post-hoc diagnostic only: labels are read AFTER the mask/loss decision and
            # never enter model inputs, threshold selection, gradients, or admission.
            audit_true=unlab.iloc[uids_np].binary_label.to_numpy(dtype=int)
            for cls in (0,1):
                sel=(pred_np==cls); accepted=sel&mask_np
                correct=accepted&(pred_np==audit_true)
                diag.append({"predicted_class":cls,"pool_n":int(sel.sum()),"accepted_n":int(accepted.sum()),
                             "acceptance_rate":float(accepted.sum()/max(sel.sum(),1)),
                             "accepted_correct_n":int(correct.sum()),
                             "accepted_precision_posthoc":float(correct.sum()/max(accepted.sum(),1)) if accepted.sum() else float("nan"),
                             "accepted_true_normal_n":int((accepted&(audit_true==0)).sum()),
                             "accepted_true_abnormal_n":int((accepted&(audit_true==1)).sum()),
                             "high_n":int((sel&(route=="high")).sum()),"low_n":int((sel&(route=="low")).sum())})

        # Validation prediction history: current epoch's prediction is written AFTER scoring prior history.
        vp,_,_=predict_val_two_views(model,val_df)
        vpred=vp.argmax(1)
        val_memory.write(np.arange(len(val_df)),vpred)
        val_metrics=evaluate(model,val_eval)
        accepted_n=sum(d["accepted_n"] for d in diag); pool_n=sum(d["pool_n"] for d in diag)
        row={"epoch":epoch+1,"train_loss":float(np.mean(epoch_losses)) if epoch_losses else float("nan"),
             "val_macro_f1":val_metrics["macro_f1"],"val_auroc":val_metrics["auroc"],
             "pseudo_acceptance_rate":accepted_n/max(pool_n,1),
             "accepted_normal":sum(d["accepted_n"] for d in diag if d["predicted_class"]==0),
             "accepted_abnormal":sum(d["accepted_n"] for d in diag if d["predicted_class"]==1),
             "tau_low_normal":tau_by_class[0],"tau_low_abnormal":tau_by_class[1]}
        history.append(row); pd.DataFrame(history).to_csv(rd/"history.csv",index=False)
        pd.DataFrame(diag).to_csv(rd/"pseudo_label_epoch_diagnostics.csv",index=False)
        score=val_metrics["macro_f1"]
        if score>best_score:
            best_score=score; bad_epochs=0
            torch.save({"model":model.state_dict(),"epoch":epoch+1,"val_metrics":val_metrics,
                        "tau_by_class":tau_by_class,"config_hash":CONFIG_HASH},best)
        else: bad_epochs+=1
        save_checkpoint(latest,model,optimizer,scaler,epoch,best_score,bad_epochs,history,
                        train_memory,val_memory,tau_by_class,calibration_done)
        print(f"{run_id} | epoch {epoch+1}/{max_epochs} | val Macro-F1={score:.4f} | accept={row['pseudo_acceptance_rate']:.3f}")
        if bad_epochs>=PATIENCE: print("Early stopping"); break

    if not best.exists(): raise RuntimeError(f"No best checkpoint produced for {run_id}")
    # Test is evaluated exactly once after checkpoint selection and threshold freeze.
    payload=torch.load(best,map_location=DEVICE,weights_only=False)
    model.load_state_dict(payload["model"])
    test_metrics=evaluate(model,make_loader(test_df,"eval",shuffle=False))
    result={"run_id":run_id,"dataset":ds,"method":method,"label_fraction":frac,
            "best_epoch":payload["epoch"],"best_val_macro_f1":payload["val_metrics"]["macro_f1"],
            "tau_low_normal":payload["tau_by_class"][0],"tau_low_abnormal":payload["tau_by_class"][1],
            "test_metrics":test_metrics,"config_hash":CONFIG_HASH}
    (rd/"test_metrics.json").write_text(json.dumps(result,indent=2))
    done.write_text(json.dumps({"completed":True,"run_id":run_id},indent=2))
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

In [8]:
import torch

run_id = "riva_5pct_lower_only_seed42"
checkpoint_path = OUT_DIR / run_id / "checkpoint_latest.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Notebook config hash:", CONFIG_HASH)
print("Checkpoint config hash:", checkpoint["config_hash"])

print("Configuration matches:",
      CONFIG_HASH == checkpoint["config_hash"])

print("Last saved epoch:", checkpoint["epoch"] + 1)
print("Saved history length:", len(checkpoint["history"]))
print("Best validation score:", checkpoint["best_score"])
print("Early stopping counter:", checkpoint["bad_epochs"])

Notebook config hash: e213e0fc343fe38f
Checkpoint config hash: e213e0fc343fe38f
Configuration matches: True
Last saved epoch: 20
Saved history length: 20
Best validation score: 0.7295363408521303
Early stopping counter: 6


## 8. Implementation smoke tests (no training)

Run these before the full matrix. They test the mask routes, memory timing, and full-batch loss normalization with synthetic tensors. They do not require the project dataset or a GPU.

In [9]:
# Test temporal memory: empty history blocks low route; three prior same-class entries enable it.
mem=PredictionMemory(n_images=2,k=MEMORY_K)
ids=np.array([0]); current=np.array([1])
stab,count=mem.stability(ids,current)
assert count[0]==0 and stab[0]==0
# append three prior predictions
for _ in range(3): mem.write(ids,current)
stab,count=mem.stability(ids,current)
assert count[0]==3 and abs(stab[0]-1.0)<1e-6
# mask tests
conf=np.array([0.99,0.75,0.75,0.75])
wp=np.array([1,1,1,1]); sp=np.array([0,1,1,1])
st=np.array([0.0,1.0,0.6,1.0]); hc=np.array([0,3,3,2])
taus={0:0.70,1:0.70}
m_fix,_=admission_mask("fixmatch",conf,wp,sp,st,hc,taus)
m_low,_=admission_mask("lower_only",conf,wp,sp,st,hc,taus)
m_ag,_=admission_mask("agreement_only",conf,wp,sp,st,hc,taus)
m_st,_=admission_mask("stability_only",conf,wp,sp,st,hc,taus)
m_cas,_=admission_mask("cas_match",conf,wp,sp,st,hc,taus)
assert m_fix.tolist()==[True,False,False,False]
assert m_low.tolist()==[True,True,True,True]
assert m_ag.tolist()==[True,True,True,True]  # low rows all agree except high row unaffected
assert m_st.tolist()==[True,True,False,False]
assert m_cas.tolist()==[True,True,False,False]
# Full unlabeled-batch denominator: one accepted loss among four = loss/4, not loss/1.
per=torch.tensor([2.0,1.0,3.0,4.0]); mask=torch.tensor([1.,0.,0.,0.])
loss=(per*mask).mean()
assert abs(loss.item()-0.5)<1e-6
print("PASS: memory warm-up, post-history stability, admission masks, and full-batch loss denominator")

PASS: memory warm-up, post-history stability, admission masks, and full-batch loss denominator


## 9. Run the full experiment matrix

This is 4 dataset/budget settings × 5 methods = 20 runs. Start only after the smoke tests pass and the manifest audit succeeds. Each run writes checkpoints and logs under `data/results/cas_match_v1/`. Interrupted runs can be resumed by rerunning the queue, provided the configuration is unchanged.

In [9]:
# RUN / RESUME ALL 20 EXPERIMENTS
# To pause: use Kernel -> Interrupt (or the stop button). The current epoch may
# replay if interrupted mid-epoch; the most recent completed-epoch checkpoint is safe.
# To resume: rerun this cell. DONE runs are skipped and incomplete runs resume.
all_results=[]
summary_path=OUT_DIR/"cas_match_summary.csv"
for ds in DATASETS:
    for frac in FRACTIONS:
        for method in METHODS:
            result=train_experiment(ds,frac,method)
            all_results.append(result)
            flat={k:v for k,v in result.items() if k!="test_metrics"}
            flat.update(result["test_metrics"])
            old=pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
            new=pd.DataFrame([flat])
            if len(old) and "run_id" in old.columns:
                old=old[old.run_id!=result["run_id"]]
            pd.concat([old,new],ignore_index=True).to_csv(summary_path,index=False)
print("Experiment queue complete:",summary_path)


cric_5pct_fixmatch_seed42 | epoch 1/100 | val Macro-F1=0.7957 | accept=0.717
cric_5pct_fixmatch_seed42 | epoch 2/100 | val Macro-F1=0.6892 | accept=0.867
cric_5pct_fixmatch_seed42 | epoch 3/100 | val Macro-F1=0.7700 | accept=0.887
cric_5pct_fixmatch_seed42 | epoch 4/100 | val Macro-F1=0.7802 | accept=0.903
cric_5pct_fixmatch_seed42 | epoch 5/100 | val Macro-F1=0.7342 | accept=0.903
cric_5pct_fixmatch_seed42 | epoch 6/100 | val Macro-F1=0.8220 | accept=0.908
cric_5pct_fixmatch_seed42 | epoch 7/100 | val Macro-F1=0.8421 | accept=0.918
cric_5pct_fixmatch_seed42 | epoch 8/100 | val Macro-F1=0.8388 | accept=0.905
cric_5pct_fixmatch_seed42 | epoch 9/100 | val Macro-F1=0.7403 | accept=0.913
cric_5pct_fixmatch_seed42 | epoch 10/100 | val Macro-F1=0.6529 | accept=0.917
cric_5pct_fixmatch_seed42 | epoch 11/100 | val Macro-F1=0.7931 | accept=0.921
cric_5pct_fixmatch_seed42 | epoch 12/100 | val Macro-F1=0.7352 | accept=0.912
cric_5pct_fixmatch_seed42 | epoch 13/100 | val Macro-F1=0.8148 | accept=0

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 305786304 vs 305786192

In [11]:
def load_checkpoint(path, model, optimizer, scaler, train_memory, val_memory):
    st = torch.load(path, map_location=DEVICE, weights_only=False)

    if st.get("config_hash") != CONFIG_HASH:
        raise ValueError(
            "Checkpoint config hash mismatch; do not resume with changed settings"
        )

    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["optimizer"])
    scaler.load_state_dict(st["scaler"])

    restore_memory(train_memory, st["train_memory"])
    restore_memory(val_memory, st["val_memory"])

    random.setstate(st["python_rng"])
    np.random.set_state(st["numpy_rng"])
    torch.set_rng_state(st["torch_rng"].cpu())

    cuda_rng = st.get("cuda_rng")
    if torch.cuda.is_available() and cuda_rng is not None:
        if isinstance(cuda_rng, torch.Tensor):
            cuda_rng = [cuda_rng]

        cuda_rng = [
            torch.as_tensor(s, dtype=torch.uint8, device="cpu").contiguous()
            for s in cuda_rng
        ]
        torch.cuda.set_rng_state_all(cuda_rng)

    return st

print("Patched checkpoint loader successfully.")

Patched checkpoint loader successfully.


In [12]:
result = train_experiment(
    ds="riva",
    frac=0.05,
    method="lower_only"
)

print(result)

Resuming riva_5pct_lower_only_seed42 at epoch 21
riva_5pct_lower_only_seed42 | epoch 21/100 | val Macro-F1=0.6968 | accept=0.941
riva_5pct_lower_only_seed42 | epoch 22/100 | val Macro-F1=0.6230 | accept=0.945
riva_5pct_lower_only_seed42 | epoch 23/100 | val Macro-F1=0.6987 | accept=0.949
riva_5pct_lower_only_seed42 | epoch 24/100 | val Macro-F1=0.6188 | accept=0.951
riva_5pct_lower_only_seed42 | epoch 25/100 | val Macro-F1=0.7115 | accept=0.936
riva_5pct_lower_only_seed42 | epoch 26/100 | val Macro-F1=0.6908 | accept=0.943
Early stopping
{'run_id': 'riva_5pct_lower_only_seed42', 'dataset': 'riva', 'method': 'lower_only', 'label_fraction': 0.05, 'best_epoch': 14, 'best_val_macro_f1': 0.7295363408521303, 'tau_low_normal': 0.8, 'tau_low_abnormal': 0.8, 'test_metrics': {'accuracy': 0.795369211514393, 'precision_abnormal': 0.702054794520548, 'abnormal_recall': 0.7282415630550622, 'specificity_normal': 0.8318840579710145, 'macro_f1': 0.7776592064600376, 'balanced_accuracy': 0.780062810513038

In [13]:

# ============================================================
# SAFE RUN / RESUME QUEUE: ALL CAS-MATCH EXPERIMENTS
# Completed runs are skipped.
# Incomplete runs resume from checkpoint_latest.pt.
# New runs start normally.
# ============================================================

import json
import shutil
import gc
import torch
import numpy as np
import random
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. SAFER CHECKPOINT LOADER
# Fixes CUDA RNG ByteTensor issue on Windows / newer PyTorch.
# train_experiment() will use this redefined function.
# ------------------------------------------------------------

def load_checkpoint(path, model, optimizer, scaler, train_memory, val_memory):
    st = torch.load(path, map_location=DEVICE, weights_only=False)

    if st.get("config_hash") != CONFIG_HASH:
        raise ValueError(
            f"Checkpoint config hash mismatch: {path}\n"
            "Do not resume with changed experiment settings."
        )

    model.load_state_dict(st["model"])
    optimizer.load_state_dict(st["optimizer"])
    scaler.load_state_dict(st["scaler"])

    restore_memory(train_memory, st["train_memory"])
    restore_memory(val_memory, st["val_memory"])

    random.setstate(st["python_rng"])
    np.random.set_state(st["numpy_rng"])
    torch.set_rng_state(st["torch_rng"].cpu())

    cuda_rng = st.get("cuda_rng")
    if torch.cuda.is_available() and cuda_rng is not None:
        if isinstance(cuda_rng, torch.Tensor):
            cuda_rng = [cuda_rng]

        cuda_rng = [
            torch.as_tensor(x, dtype=torch.uint8, device="cpu").contiguous()
            for x in cuda_rng
        ]
        torch.cuda.set_rng_state_all(cuda_rng)

    return st


# ------------------------------------------------------------
# 2. SAFETY CHECKS
# ------------------------------------------------------------

assert ROOT.exists(), f"Project root missing: {ROOT}"
assert OUT_DIR.exists(), f"Output directory missing: {OUT_DIR}"
assert callable(train_experiment), "train_experiment() is not defined."
assert callable(run_name), "run_name() is not defined."

MIN_FREE_GB = 2.5
summary_path = OUT_DIR / "cas_match_summary.csv"
queue_log_path = OUT_DIR / "queue_progress.csv"

def free_gb():
    return shutil.disk_usage(ROOT).free / (1024 ** 3)

print("=" * 65)
print("CAS-MATCH RESUMABLE EXPERIMENT QUEUE")
print("=" * 65)
print("Project:", ROOT)
print("Output:", OUT_DIR)
print("Device:", DEVICE)
print("Free disk space:", f"{free_gb():.2f} GB")
print("Configuration hash:", CONFIG_HASH)
print("Total planned runs:", len(DATASETS) * len(FRACTIONS) * len(METHODS))

if free_gb() < MIN_FREE_GB:
    raise RuntimeError(
        f"Only {free_gb():.2f} GB free. "
        f"Free at least {MIN_FREE_GB} GB before starting."
    )


# ------------------------------------------------------------
# 3. QUEUE ALL EXPERIMENTS
# train_experiment() itself handles DONE and checkpoint resume.
# ------------------------------------------------------------

queue_records = []
all_results = []

for ds in DATASETS:
    for frac in FRACTIONS:
        for method in METHODS:

            run_id = run_name(ds, frac, method)
            run_dir = OUT_DIR / run_id

            latest = run_dir / "checkpoint_latest.pt"
            best = run_dir / "checkpoint_best.pt"
            done = run_dir / "DONE.json"
            metrics_file = run_dir / "test_metrics.json"

            print("\n" + "=" * 65)
            print("RUN:", run_id)

            # Classify the run before calling train_experiment().
            if done.exists() and metrics_file.exists():
                status = "COMPLETED - SKIP"
            elif latest.exists():
                status = "INCOMPLETE - RESUME"
            else:
                status = "NOT STARTED - NEW RUN"

            print("Status:", status)
            print("Free disk:", f"{free_gb():.2f} GB")

            if free_gb() < MIN_FREE_GB:
                raise RuntimeError(
                    f"Disk-space safety stop before {run_id}. "
                    "Free space, then rerun this same queue cell."
                )

            # Record that this run is about to be processed.
            queue_records.append({
                "run_id": run_id,
                "status": status,
                "started_at": pd.Timestamp.now().isoformat(),
            })
            pd.DataFrame(queue_records).to_csv(queue_log_path, index=False)

            # This function skips completed runs and resumes
            # incomplete runs using checkpoint_latest.pt.
            result = train_experiment(
                ds=ds,
                frac=frac,
                method=method
            )

            all_results.append(result)

            # Update summary after EVERY successful run.
            flat = {
                k: v for k, v in result.items()
                if k != "test_metrics"
            }
            flat.update(result["test_metrics"])

            old = (
                pd.read_csv(summary_path)
                if summary_path.exists()
                else pd.DataFrame()
            )

            if len(old) and "run_id" in old.columns:
                old = old[old["run_id"] != run_id]

            updated = pd.concat(
                [old, pd.DataFrame([flat])],
                ignore_index=True
            )
            updated.to_csv(summary_path, index=False)

            print("\nFINISHED:", run_id)
            print("Best validation Macro-F1:", result["best_val_macro_f1"])
            print("Test Macro-F1:", result["test_metrics"]["macro_f1"])
            print("Test AUROC:", result["test_metrics"]["auroc"])

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ------------------------------------------------------------
# 4. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("QUEUE FINISHED")
print("=" * 65)

completed = 0
incomplete = []

for ds in DATASETS:
    for frac in FRACTIONS:
        for method in METHODS:
            run_id = run_name(ds, frac, method)
            run_dir = OUT_DIR / run_id

            if (
                (run_dir / "DONE.json").exists()
                and (run_dir / "test_metrics.json").exists()
            ):
                completed += 1
            else:
                incomplete.append(run_id)

print(f"Completed: {completed}/20")
print(f"Remaining: {len(incomplete)}")

if incomplete:
    print("Still incomplete:")
    for x in incomplete:
        print(" -", x)
else:
    print("All 20 experiments have completed.")

print("Summary:", summary_path)
print("Queue log:", queue_log_path)


CAS-MATCH RESUMABLE EXPERIMENT QUEUE
Project: C:\Users\nanda\Documents\cervical-semi-sl
Output: C:\Users\nanda\Documents\cervical-semi-sl\data\results\cas_match_v1
Device: cuda
Free disk space: 0.54 GB
Configuration hash: e213e0fc343fe38f
Total planned runs: 20


RuntimeError: Only 0.54 GB free. Free at least 2.5 GB before starting.

## Run control: pause, stop, and resume\n
- **Run all:** execute the `RUN / RESUME ALL 20 EXPERIMENTS` cell.\n
- **Pause/stop now:** use Jupyter's Kernel → Interrupt (or the stop-square button).\n
- **Checkpoint granularity:** the latest resumable checkpoint is written at the end of each epoch. If interrupted mid-epoch, that incomplete epoch is replayed from its beginning on resume; completed epochs are retained.\n
- **Resume:** rerun the run-all cell. Completed runs are skipped using `DONE.json` and their full `test_metrics.json` is restored into the summary. An interrupted run resumes from `checkpoint_latest.pt`.\n
- **Do not change** training, augmentation, calibration, or split settings when resuming. A configuration hash now includes augmentation settings and the exact split-manifest file hashes; incompatible checkpoints are rejected.\n
- The best-validation checkpoint is kept separately from the latest training checkpoint. Test evaluation occurs only after training/checkpoint selection and is not used for resuming model selection.\n

## 10. Ablation comparison and reporting

Compare methods within each dataset and label budget. Report Macro-F1, AUROC, balanced accuracy, Abnormal recall, Normal specificity, per-class F1, confusion matrix counts, selected class thresholds, calibration support/precision, and pseudo-label acceptance diagnostics. Interpret one-seed differences cautiously; repeat shortlisted settings with multiple seeds before claiming robust gains.

In [ ]:
summary_path=OUT_DIR/"cas_match_summary.csv"
if summary_path.exists():
    summary=pd.read_csv(summary_path)
    display(summary.sort_values(["dataset","label_fraction","method"]))
    metric_cols=["macro_f1","auroc","balanced_accuracy","abnormal_recall","specificity_normal","normal_f1","abnormal_f1"]
    pivot=summary.pivot_table(index=["dataset","label_fraction"],columns="method",values=metric_cols)
    display(pivot)
else:
    print("No completed runs yet.")

## Important methodological notes and limitations

- This is a single-seed implementation by default. Use at least three seeds, preferably five, for inferential claims.
- Validation pseudo-label precision is an offline calibration statistic. Validation images are never included in the training loss.
- The calibration threshold is selected after warm-up using validation predictions and validation labels. This is a form of supervised validation calibration, not unlabeled-only threshold adaptation. State it explicitly in any paper.
- For `agreement_only` and `cas_match`, the validation agreement signal is computed from one sampled strong view per image at calibration time. The training agreement signal is computed from the actual weak/strong views used in that batch.
- Temporal memory is per image ID, with one weak prediction written per image per epoch. Low-confidence routes are unavailable until three previous observations exist.
- The `lower_only`, `agreement_only`, and `stability_only` methods all include the same high-confidence FixMatch route plus their specified low-confidence route. This makes the ablation ladder nested and interpretable.
- The test set is evaluated only after selecting the best checkpoint and freezing thresholds. Do not use test metrics to alter thresholds or hyperparameters.
- Phase 11 files and outputs are not read for training and are not overwritten. All new outputs go to `data/results/cas_match_v1/`.